# EXO_03_GEOMETRY_PROBE — Colab (U03)

Exécute **`geometry_probe_u03.py`** en headless sur `environment_1.blend` et écrit un **JSON** (pas de parse de stdout Blender).

**Prérequis :** Drive monté, `OUT_PREMIUM_SCENE/environment_1.blend` présent, Blender installé (ex. `/opt/blender-local/blender` après `setup_blender_deps.py` ou équivalent).

**Référence GitHub :** `03_SCENOGRAPHY_DOCK/CODEBASE/geometry_probe_u03.py`

## 1. Monter Drive + chemins

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive", force_remount=True)

DRIVE_ROOT = Path("/content/drive/MyDrive/EXODUS_V2")
U03 = DRIVE_ROOT / "03_SCENOGRAPHY_DOCK"
CODEBASE = U03 / "CODEBASE"
OUT_SCENE = U03 / "OUT_PREMIUM_SCENE"
ENV_BLEND = OUT_SCENE / "environment_1.blend"
PROBE_SCRIPT = CODEBASE / "geometry_probe_u03.py"
OUT_JSON = OUT_SCENE / "U03_geometry_probe.json"

print("DRIVE_ROOT  :", DRIVE_ROOT)
print("ENV_BLEND   :", ENV_BLEND, "->", ENV_BLEND.exists())
print("PROBE_SCRIPT:", PROBE_SCRIPT, "->", PROBE_SCRIPT.exists())

## 2. (Optionnel) Synchroniser le CODEBASE depuis GitHub

In [ ]:
import subprocess
import shutil

REPO = Path("/tmp/exodus-geometry-probe")
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(
    ["git", "clone", "--depth", "1", "https://github.com/kioka8877-ux/EXODUS-V2.git", str(REPO)],
    check=True,
)
src = REPO / "03_SCENOGRAPHY_DOCK" / "CODEBASE" / "geometry_probe_u03.py"
CODEBASE.mkdir(parents=True, exist_ok=True)
if src.exists():
    shutil.copy2(src, PROBE_SCRIPT)
    print("OK — geometry_probe_u03.py copié depuis main")
else:
    print("ERREUR — script absent dans le clone")

## 3. Chemin Blender

In [ ]:
import os

BLENDER_CANDIDATES = [
    Path("/opt/blender-local/blender"),
    Path("/usr/bin/blender"),
]
BLENDER_BIN = next((p for p in BLENDER_CANDIDATES if p.exists()), None)
if BLENDER_BIN is None:
    raise FileNotFoundError("Installe Blender (voir setup_blender_deps.py à la racine EXODUS_V2)")
os.chmod(str(BLENDER_BIN), 0o755)
print("Blender:", BLENDER_BIN)

## 4. Lancer la probe (écrit le JSON sur Drive)

In [ ]:
import json
import subprocess

if not ENV_BLEND.exists():
    raise FileNotFoundError(f"Manquant: {ENV_BLEND}")
if not PROBE_SCRIPT.exists():
    raise FileNotFoundError(f"Manquant: {PROBE_SCRIPT} — lance la cellule git clone ci-dessus")

cmd = [
    str(BLENDER_BIN),
    "--background",
    str(ENV_BLEND),
    "--python",
    str(PROBE_SCRIPT),
    "--",
    "--output",
    str(OUT_JSON),
]
print(" ".join(cmd[:6]), "...")
r = subprocess.run(cmd, capture_output=True, text=True, timeout=180)
print("returncode:", r.returncode)
if r.stdout:
    print(r.stdout[-2000:])
if r.stderr:
    print("STDERR:", r.stderr[-1500:])

if OUT_JSON.exists():
    report = json.loads(OUT_JSON.read_text(encoding="utf-8"))
    print("\n=== RAPPORT JSON ===")
    print(json.dumps(report, indent=2, ensure_ascii=False)[:8000])
    print("\n... [tronqué si très long]")
    print("\nstatus:", report.get("status"))
else:
    print("ERREUR: JSON non créé")


## 5. Lecture du fichier (alternative)

Si la cellule précédente a réussi, le rapport est aussi lisible depuis le fichier :

In [ ]:
from IPython.display import display, JSON
if OUT_JSON.exists():
    display(JSON(json.loads(OUT_JSON.read_text(encoding="utf-8"))))
else:
    print("Pas de fichier:", OUT_JSON)